In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ==========================================
# 1. RESIDUAL BLOCK DEFINITION
# ==========================================
def residual_block(x, filters, kernel_size=3):
    shortcut = x

    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)

    x = layers.Conv1D(filters, kernel_size, padding='same')(x)
    x = layers.BatchNormalization()(x)

    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv1D(filters, 1, padding='same')(shortcut)

    x = layers.Add()([x, shortcut])
    x = layers.Activation('relu')(x)
    return x

In [3]:
# ==========================================
# 2. LOAD BOTH DATASETS (SIGNAL + BMI)
# ==========================================
print("Loading Deep Learning Dataset (Signal + BMI)...")
X_signals = np.load("X_signals.npy")  # Shape: (2150, 100, 2)
X_bmi = np.load("X_bmi.npy")          # Shape: (2150,)
y_glucose = np.load("y_glucose.npy")  # Shape: (2150,)

# Split the data identically for both inputs
X_sig_train, X_sig_test, X_bmi_train, X_bmi_test, y_train, y_test = train_test_split(
    X_signals, X_bmi, y_glucose, test_size=0.2, random_state=42
)

Loading Deep Learning Dataset (Signal + BMI)...


In [4]:
# ==========================================
# 3. BUILD THE TWO-HEADED RESNET
# ==========================================
# --- HEAD 1: The ResNet Signal Processor ---
input_signal = layers.Input(shape=(100, 2), name="signal_input")

x = layers.Conv1D(32, kernel_size=7, padding='same')(input_signal)
x = layers.BatchNormalization()(x)
x = layers.Activation('relu')(x)
x = layers.MaxPooling1D(pool_size=2)(x)

x = residual_block(x, filters=32)
x = residual_block(x, filters=64)
x = layers.MaxPooling1D(pool_size=2)(x)
x = residual_block(x, filters=128)

x = layers.GlobalAveragePooling1D()(x) # Compress signal features safely

# --- HEAD 2: The BMI Processor ---
input_bmi = layers.Input(shape=(1,), name="bmi_input")
y_branch = layers.Dense(16, activation='relu')(input_bmi)
y_branch = layers.BatchNormalization()(y_branch)

# --- THE MERGE: Combine Signal Math with BMI Context ---
merged = layers.Concatenate()([x, y_branch])

# Final decision layers using the combined data
z = layers.Dense(64, activation='relu')(merged)
z = layers.Dropout(0.3)(z)
z = layers.Dense(32, activation='relu')(z)

# Output Layer
output = layers.Dense(1, activation='linear', name="glucose_output")(z)

# Compile Model
model = Model(inputs=[input_signal, input_bmi], outputs=output)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

In [5]:
# ==========================================
# 4. TRAIN THE MODEL
# ==========================================
print("\nTraining the Two-Headed ResNet Model...")

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=0.00001)
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

history = model.fit(
    x=[X_sig_train, X_bmi_train], 
    y=y_train,
    validation_data=([X_sig_test, X_bmi_test], y_test),
    epochs=150,
    batch_size=32,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)


Training the Two-Headed ResNet Model...
Epoch 1/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 22s 174ms/step - loss: 11135.7363 - mae: 97.6159 - val_loss: 10094.8242 - val_mae: 94.2716 - learning_rate: 5.0000e-04
Epoch 2/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1648.7111 - mae: 31.9345 - val_loss: 5098.7539 - val_mae: 62.3681 - learning_rate: 5.0000e-04
Epoch 3/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 1386.3832 - mae: 29.3988 - val_loss: 2856.0920 - val_mae: 40.8766 - learning_rate: 5.0000e-04
Epoch 4/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1411.3939 - mae: 29.8400 - val_loss: 1664.2010 - val_mae: 28.9780 - learning_rate: 5.0000e-04
Epoch 5/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1259.8619 - mae: 28.1271 - val_loss: 1188.7090 - val_mae: 26.9809 - learning_rate: 5.0000e-04
Epoch 6/150
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 1241.9891 - mae: 28.0143 - val_loss: 1156.8210 - val_mae: 27.0630 - learning_rate: 5.0000e-04
Epoch 7/150
54/54 ━━━━━━━━━━━━━━━━━━

In [7]:
# ==========================================
# 5. EVALUATE
# ==========================================
print("\n==================================================")
print("🏆 TWO-HEADED RESNET RESULTS (SIGNAL + BMI)")
print("==================================================")

predictions = model.predict([X_sig_test, X_bmi_test]).flatten()
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"➡️ Mean Absolute Error (MAE): {mae:.2f} mg/dL")
print(f"➡️ Root Mean Squared Error (RMSE): {rmse:.2f} mg/dL")
print(f"➡️ R-Squared (R2):            {r2:.2f}")


🏆 TWO-HEADED RESNET RESULTS (SIGNAL + BMI)
14/14 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
➡️ Mean Absolute Error (MAE): 22.57 mg/dL
➡️ Root Mean Squared Error (RMSE): 28.42 mg/dL
➡️ R-Squared (R2):            0.34


In [8]:
model.save("Two_Head_Resnet.h5")
print("\nModel saved successfully as 'Two_Head_Resnet.h5'")


Model saved successfully as 'Two_Head_Resnet.h5'
